# Tutorial: Test set - Baseline TalentCLEF 2026 - Task A

In this notebook, we provide the Test Set baseline for TalentCLEF 2026 - Task A. This document includes downloading the[Task A dataset](https://doi.org/10.5281/zenodo.17625261), applying a multilingual embedding model as a baseline to generate .trec files, which will then be compressed and uploaded to the Codalab platform.


-----------------------------
TalentCLEF is an initiative to advance Natural Language Processing (NLP) in Human Capital Management (HCM). It aims to create a public benchmark for model evaluation and promote collaboration to develop fair, multilingual, and flexible systems that improve Human Resources (HR) practices across different industries.

The second edition of TalentCLEF shared task’s will be part of the [Conference and Labs of the Evaluation Forum (CLEF)](https://clef2026.clef-initiative.eu/), scheduled to be held in Jena, Germany, in 2026. If you are interested in registering, you can find registration form [here](https://clef-labs-registration.dipintra.it/)


<img src="https://github.com/TalentCLEF/talentclef/blob/main/logo_talentclef.png?raw=true" alt="TalentCLEF logo" width="200"/>
<img src="https://talentclef.github.io/talentclef/docs/talentclef-2026/workshop/logo_clef_jena.svg" alt="CLEF2026 logo" width="150"/>

## Imports

In [ ]:
import json

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
!pip install codecarbon
from codecarbon import EmissionsTracker

## Download Task A files

First, let's download the Task A from Zenodo.


In [ ]:
# Download
!wget https://zenodo.org/records/19652670/files/TaskA.zip
!unzip TaskA.zip -d taskA

### Baseline

Define language directionalities (queries-documents):

In [3]:
language_pairs = ["en-en","es-es","en-es"]

The baseline model is [`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2)

In [4]:
models = ["sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"]

Function to load data:

In [5]:
def load_text_corpus(path, id_col="c_id", encoding="utf-8"):
    """
    Load text files from a directory into a pandas DataFrame.

    Each file in the directory is treated as a document. The file name is used
    as the document identifier, and the file content is stored as text.

    Parameters
    ----------
    path : str
        Path to the directory containing the text files.
    id_col : str, optional
        Name of the column used as the document identifier
        (e.g., 'c_id', 'q_id'). Default is 'c_id'.
    encoding : str, optional
        Text encoding used to read the files. Default is 'utf-8'.

    Returns
    -------
    pd.DataFrame
        A DataFrame with two columns:
        - id_col: document identifier (file name)
        - text: document content
    """
    records = []

    for filename in os.listdir(path):
        file_path = os.path.join(path, filename)

        if os.path.isfile(file_path):
            with open(file_path, "r", encoding=encoding, errors="ignore") as f:
                text = f.read()

            records.append({
                id_col: filename,
                "text": text
            })

    return pd.DataFrame(records)

Apply the model and save results:

In [ ]:
import os
emissions = {}

for model_name in models:
  tracker = EmissionsTracker()
  tracker.start_task(model_name)
  # Download and load embedding model
  model = SentenceTransformer(model_name)
  for language_pair in language_pairs:
    query_lang, corpus_lang = language_pair.split("-")
    # Read queries and corpus elements for the specific language
    queries = f"/content/taskA/TaskA/test/{query_lang}/queries"
    corpus_elements = f"/content/taskA/TaskA/test/{corpus_lang}/corpus"
    queries = load_text_corpus(queries, id_col="q_id")
    corpus =  load_text_corpus(corpus_elements, id_col="c_id")
    # Get ids, strings and generate a mapping dictionary for queries
    queries_ids = queries.q_id.to_list()
    queries_texts = queries.text.to_list()
    map_queries = dict(zip(queries_ids,queries_texts))
    # Get ids, strings and generate a mapping dictionary for corpus elements
    corpus_ids = corpus.c_id.to_list()
    corpus_texts = corpus.text.to_list()
    map_corpus = dict(zip(queries_ids,queries_texts))
    # Encode queries and corpus elements with the baseline model.
    query_embeddings = model.encode(queries_texts, convert_to_tensor=True)
    corpus_embeddings = model.encode(corpus_texts, convert_to_tensor=True)

    # Compute similarities between query and corpus element embeddings
    similarities = util.cos_sim(query_embeddings, corpus_embeddings).cpu().numpy()

    # Process results and prepare output file
    results = []
    for q_idx, q_id in enumerate(queries_ids):
        sorted_indices = np.argsort(-similarities[q_idx])  # Decrease order
        for rank, c_idx in enumerate(sorted_indices):
            doc_id = corpus_ids[c_idx]
            score = similarities[q_idx, c_idx]
            results.append(f"{str(q_id)} Q0 {str(doc_id)} {rank+1} {score:.4f} baseline_model")

    # Save the predictions in a trecfile. Follow the naming guidelines
    with open(f"run_{language_pair}_testbaseline-{model_name.split('/')[1]}.trec", "w", encoding="utf-8") as f:
        f.write("\n".join(results))
    pass
  emissions[model_name]: float = dict(tracker.stop_task(model_name).values)

json.dump(emissions, open("./emissions.json", "w"), ensure_ascii=False, indent=4)

Zip the results that will be uploaded into the [Task A Codabench](https://www.codabench.org/competitions/14226)

In [ ]:
!zip taskA_testset_baseline.zip run_* emissions.json